# Coding Task: Production-Grade RAG with Self-Evaluation

**Role:** AI Engineer
**Time budget:** ~35 minutes (aim for a working end-to-end v1 by minute 25, then improve)
**Libraries allowed:** `openai`, `langchain` / `langgraph`, `sentence-transformers`, `rank_bm25`, `numpy`, `pandas`, standard lib. Any vector store is fine (FAISS, Chroma, in-memory).
**Not allowed:** AI coding assistants.

You have 35 minutes, so scope matters more than polish. **A working end-to-end system with honest metrics beats a half-built "perfect" one.**

## Scenario

You are the first AI engineer on a small product that answers employee questions about an internal policy knowledge base. Leadership just ran a pilot and users complained that the system "sounds confident but sometimes makes things up."

Your job in this exercise is to:

1. Build a working RAG pipeline over the provided corpus.
2. Make it answer with **inline citations** that point to specific source chunks.
3. Add a **self-evaluation layer** that flags answers that are not grounded in retrieved context.
4. Run it against the provided eval set and **report honest metrics**, not just accuracy.

You are expected to make design choices and defend them in comments. There is no single correct answer. We care more about your tradeoffs than about hitting some benchmark.

## What "done" looks like

You will implement three things:

### A. `class RAGSystem`
A class with these methods:

```python
rag = RAGSystem(documents: list[dict])      # ingests and indexes
rag.answer(question: str) -> AnswerResult    # retrieves + generates + cites
```

where `AnswerResult` is a dataclass (or dict) with at minimum:
- `question: str`
- `answer: str` (with inline citation markers like `[1]`, `[2]`)
- `retrieved_chunks: list[dict]` (id, text, score)
- `citations: list[str]` (chunk ids the answer actually uses)
- `grounded: bool` (from the self-eval layer, see C)
- `groundedness_reason: str` (one-sentence explanation)

### B. A retrieval strategy of your choice
Use dense, sparse, or hybrid. **Briefly justify your choice in a comment.** We'll ask you why during the walk-through.

### C. A self-evaluation layer
Given the final answer and the retrieved chunks, decide whether the answer is grounded in the retrieved context. This is **not** the same as "is the answer correct". It's "does the answer follow from the retrieved chunks?"

You may use an LLM-as-judge, a rule-based check, or a hybrid. Justify briefly.

### D. Run the eval set
Run `rag.answer(q)` on every question in `EVAL_SET` below and produce a small results table. Report at least:

- **Answer rate** (non-empty answers / total).
- **Grounded rate** (your self-eval says grounded / total).
- **Retrieval hit rate** (a gold chunk id appears in the top-k retrieved).
- **At least one additional metric of your choosing**, with a one-sentence justification.

### What NOT to do
- Don't try to fine-tune anything.
- Don't build a UI.
- Don't chase 100% on the eval set. 5 of 8 with honest metrics and a clear explanation of the 3 misses is a stronger result than 8 of 8 with no introspection.

## 1. Setup

In [ ]:
!pip install -q openai==1.55.3 sentence-transformers==3.3.1 rank_bm25==0.2.2 numpy pandas

In [ ]:
import os
import json
import re
import time
from dataclasses import dataclass, field, asdict
from typing import Any
import numpy as np
import pandas as pd
from openai import OpenAI

API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not API_KEY:
    API_KEY = input("Paste your OpenAI API key: ").strip()

client = OpenAI(api_key=API_KEY)

# gpt-4o-mini is fast, cheap, and good enough for this exercise.
# Feel free to change this if you want, but justify the switch.
MODEL = "gpt-4o-mini"


## 2. Provided Corpus

A small synthetic internal knowledge base for a fictional company. Each document has an id, a title, and body text. The corpus is intentionally small so you can focus on pipeline design, not scale.

In [ ]:
DOCUMENTS = [
    {
        "id": "POL-001",
        "title": "Remote Work Policy",
        "text": (
            "Employees may work remotely up to three days per week by default. "
            "Engineering and design roles may request fully remote status with "
            "manager approval and at least six months of tenure. All remote "
            "workers are required to attend the company-wide on-site meetings "
            "held twice per year, in March and October. Travel costs for these "
            "on-sites are fully reimbursed up to an economy fare cap of 800 USD "
            "per round trip. Seat upgrades are not reimbursed."
        ),
    },
    {
        "id": "POL-002",
        "title": "Expense Reimbursement",
        "text": (
            "Business-related expenses are reimbursed when submitted within "
            "thirty days of purchase. Meals during business travel are capped "
            "at 75 USD per day. Alcohol is not reimbursable. Client dinners "
            "require pre-approval from a department head when the per-person "
            "cost exceeds 100 USD. All receipts must be uploaded to the Expenses "
            "portal; screenshots of credit card statements are not sufficient."
        ),
    },
    {
        "id": "POL-003",
        "title": "Parental Leave",
        "text": (
            "Primary caregivers are entitled to sixteen weeks of paid parental "
            "leave. Secondary caregivers receive eight weeks. Leave must be "
            "taken within the first twelve months following the birth or "
            "adoption of a child. Employees must notify HR at least sixty days "
            "in advance when possible. Leave may be taken continuously or split "
            "into at most two separate blocks."
        ),
    },
    {
        "id": "POL-004",
        "title": "Laptop and Equipment",
        "text": (
            "New hires receive a standard laptop within their first week. "
            "Engineers may request an upgraded configuration with manager "
            "approval. Equipment refresh happens on a three-year cycle. "
            "Employees are responsible for returning all equipment on their "
            "last day of employment. Lost or damaged equipment may be charged "
            "to the employee up to a maximum of 500 USD."
        ),
    },
    {
        "id": "POL-005",
        "title": "Security and VPN",
        "text": (
            "All employees must connect to the corporate VPN when accessing "
            "internal systems from outside the office. Two-factor authentication "
            "is mandatory for all company accounts. Passwords must be at least "
            "fourteen characters, rotated every ninety days, and never reused "
            "across services. Security incidents must be reported to the "
            "security team within four hours of discovery."
        ),
    },
    {
        "id": "POL-006",
        "title": "Conference and Training Budget",
        "text": (
            "Each employee has an annual learning budget of 2000 USD. This may "
            "be used for conferences, courses, books, or certifications. "
            "Budget does not roll over between calendar years. Conference "
            "travel falls under the standard travel policy. Certifications "
            "required by the employee's role are funded separately and do not "
            "count against the learning budget."
        ),
    },
    {
        "id": "POL-007",
        "title": "Performance Reviews",
        "text": (
            "Formal performance reviews occur twice per year, in June and "
            "December. Reviews include a self-assessment, peer feedback from "
            "at least three colleagues, and a manager evaluation. Promotions "
            "are announced in the first week of July and January. Compensation "
            "adjustments take effect in the following pay cycle."
        ),
    },
    {
        "id": "POL-008",
        "title": "Data Handling and Privacy",
        "text": (
            "Customer data must never be copied to personal devices. Any "
            "export of data from production systems requires approval from "
            "the Data Protection Officer and must be logged. Employees who "
            "handle personally identifiable information complete mandatory "
            "annual privacy training. Violations are treated as a serious "
            "disciplinary matter."
        ),
    },
    {
        "id": "POL-009",
        "title": "Office Hours and Core Collaboration Windows",
        "text": (
            "Employees are expected to be available for synchronous "
            "collaboration between 10:00 and 16:00 in their local time zone. "
            "Meetings scheduled outside these hours require explicit opt-in "
            "from attendees. Cross-continent meetings should rotate times "
            "fairly between regions and not consistently inconvenience the "
            "same team."
        ),
    },
    {
        "id": "POL-010",
        "title": "Sick Leave",
        "text": (
            "Employees accrue sick leave at a rate of one day per month, up to "
            "a maximum carryover of twenty days. Sick leave can also be used "
            "to care for an immediate family member. A doctor's note is "
            "required for absences exceeding three consecutive days. Unused "
            "sick leave is not paid out upon termination."
        ),
    },
]

print(f"Loaded {len(DOCUMENTS)} documents.")


## 3. Provided Eval Set

Each entry has a question, the chunk id(s) that should ground the answer (`gold_chunk_ids`), and a short description of what a good answer contains. Some entries are **deliberately tricky**: one has no answer in the corpus, and one requires combining two documents. Handle them honestly.

In [ ]:
EVAL_SET = [
    {
        "qid": "Q1",
        "question": "How many weeks of paid parental leave does a primary caregiver get?",
        "gold_chunk_ids": ["POL-003"],
        "expected_contains": ["16", "sixteen"],
    },
    {
        "qid": "Q2",
        "question": "What is the daily meal cap during business travel?",
        "gold_chunk_ids": ["POL-002"],
        "expected_contains": ["75"],
    },
    {
        "qid": "Q3",
        "question": "How often are performance reviews held and when are promotions announced?",
        "gold_chunk_ids": ["POL-007"],
        "expected_contains": ["June", "December", "July", "January"],
    },
    {
        "qid": "Q4",
        "question": (
            "I'm an engineer attending the March on-site. My round-trip flight is 1100 USD. "
            "How much of it will the company reimburse?"
        ),
        "gold_chunk_ids": ["POL-001"],
        "expected_contains": ["800"],
    },
    {
        "qid": "Q5",
        "question": "What is the company dental insurance coverage?",
        "gold_chunk_ids": [],  # not in corpus
        "expected_contains": [],  # the right answer is "I don't know / not in the corpus"
        "note": "This question has no answer in the corpus. A good system should say so.",
    },
    {
        "qid": "Q6",
        "question": (
            "I have a conference in May that costs 1500 USD for the ticket plus a 900 USD flight. "
            "Do both fit in my learning budget?"
        ),
        "gold_chunk_ids": ["POL-006", "POL-001"],
        "expected_contains": ["2000", "travel"],
        "note": "Requires combining two documents: learning budget vs. travel policy.",
    },
    {
        "qid": "Q7",
        "question": "What are the password requirements?",
        "gold_chunk_ids": ["POL-005"],
        "expected_contains": ["14", "fourteen", "90", "ninety"],
    },
    {
        "qid": "Q8",
        "question": "If I lose my company laptop, how much could I be charged?",
        "gold_chunk_ids": ["POL-004"],
        "expected_contains": ["500"],
    },
]

print(f"Eval set: {len(EVAL_SET)} questions.")


## 4. Your Work

Below is a scaffold. Fill it in. Add cells as needed.

### Notes
- Keep the corpus as-is or chunk it further. Justify your choice in a comment.
- The LLM has no memory between calls. You must include retrieved context in every generation call.
- Citations in the final answer should look like `[POL-003]`, not like `[1]`. This is small but it matters for the grader.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class AnswerResult:
    question: str
    answer: str
    retrieved_chunks: list = field(default_factory=list)  # list[{"id", "text", "score"}]
    citations: list = field(default_factory=list)         # list[str], chunk ids referenced
    grounded: bool = False
    groundedness_reason: str = ""

    def to_dict(self):
        return asdict(self)


In [ ]:
class RAGSystem:
    """
    Build me.

    Design choices you will need to defend:
      - Chunking strategy (or no chunking).
      - Retrieval: sparse vs dense vs hybrid.
      - Top-k value and why.
      - Prompt design for grounded generation with citations.
      - Self-evaluation mechanism for groundedness.
    """

    def __init__(self, documents: list):
        self.documents = documents
        # TODO: build your index here.
        pass

    def retrieve(self, query: str, top_k: int = 4) -> list:
        """Return a list of {"id", "text", "score"} for the top_k most relevant chunks."""
        # TODO
        raise NotImplementedError

    def generate_answer(self, question: str, chunks: list) -> str:
        """Prompt the LLM to answer using ONLY the provided chunks, with inline [POL-xxx] citations."""
        # TODO
        raise NotImplementedError

    def evaluate_groundedness(self, question: str, answer: str, chunks: list) -> tuple:
        """Return (grounded: bool, reason: str)."""
        # TODO
        raise NotImplementedError

    def answer(self, question: str) -> AnswerResult:
        # TODO: wire the pieces together.
        raise NotImplementedError


### Design justification (write me)

Use this cell to briefly state, in 3-6 bullets, your choices and why. The interviewer will read this. Examples of things to say something about:

- Why this retrieval strategy.
- Why this chunking (or none).
- How your groundedness check works and what it will miss.
- What you would do next if you had another hour.

_Write your justification here._

-
-
-


## 5. Run the Eval Set and Report Metrics

Run your system against every question in `EVAL_SET` and produce a results table plus summary metrics. Required metrics:

- **Answer rate**
- **Grounded rate** (your self-eval)
- **Retrieval hit rate** (any `gold_chunk_ids` appears in top-k retrieved)
- **One more metric of your choosing** (justify in a comment or markdown cell)

In [ ]:
# TODO: Build `rag`, run every question, print a results DataFrame and summary metrics.

rag = RAGSystem(DOCUMENTS)

rows = []
for item in EVAL_SET:
    # TODO
    pass

# Expected final output: a DataFrame with one row per question and a short summary print
# of your chosen metrics. Make it readable.


## 6. Stretch Goals (only if time)

Pick at most one. Flag it clearly.

**A. Hybrid retrieval.** If you shipped dense or sparse alone, add the other and combine with Reciprocal Rank Fusion. Re-run the eval and compare.

**B. Adversarial refusal.** Add a pre-generation check: if the top retrieval score is below some threshold, skip generation and return a calibrated "I don't know, this does not appear to be covered in the policy documents." Show that this changes behavior on Q5.

**C. Answer attribution audit.** For every claim in the final answer, verify it appears (semantically, not just textually) in at least one cited chunk. Mark unattributed claims in the output. This is a stronger version of the groundedness check.

In [ ]:
# Optional stretch-goal implementation goes here.
